# chem-test - N_chem=1 single-food parity gate (milestone 1)

The `chem` branch introduces vector-valued resource ("chem") machinery.
**Milestone 1's contract:** at `N_chem == 1` the model reproduces the
pre-change single-food dynamics **bit for bit**.

This notebook replays a fixed-seed initial state on the current build and
compares it, field by field, against the golden snapshot captured from the
pre-change build (`tests/data/chem_parity_golden.npz`). It prints a clear
**PASS / FAIL**. See `Docs/plan-chem.md` for the pinned design (sparse
agent-local storage + the vector->`f` mapping).

In [1]:
import os, sys
ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, 'C', 'evoca.h')):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, os.path.join(ROOT, 'tests'))
sys.path.insert(0, os.path.join(ROOT, 'python'))
import numpy as np
import chem_parity as cp
from evoca_py import EvoCA
print('golden:', cp.GOLDEN_PATH)
print('config:', cp.CFG)
print('N =', cp.N, ' steps =', cp.N_STEPS, ' checkpoints =', cp.CHECKPOINTS)

golden: /private/tmp/evoca_tier2/tests/data/chem_parity_golden.npz
config: {'food_inc': 0.02, 'm_scale': 3.0, 'gdiff': 0.05, 'tax': 0.02, 'mu_lut': 0.01, 'mu_egene': 0.005, 'mu_egenome': 0.01, 'p_dup_egene': 1.0}
N = 48  steps = 200  checkpoints = (50, 100, 150, 200)


## 1. Confirm the chem build defaults to N_chem=1
and that unsupported universe sizes are surfaced (clamped + warned), since
reactions are milestones 2-3.

In [2]:
sim = EvoCA()
sim.init(N=8, food_inc=0.01, m_scale=1.0)
print('default N_chem      =', sim.get_n_chem())
print('chem_energy[0]      =', sim.get_chem_energy(0))
eff = sim.set_n_chem(4)   # unsupported at milestone 1 -> clamps to 1
print('set_n_chem(4) -> eff =', eff, '(clamped)')
assert sim.get_n_chem() == 1 and sim.get_chem_energy(0) == 1.0

default N_chem      = 1
chem_energy[0]      = 1.0
set_n_chem(4) -> eff = 1 (clamped)


evoca_set_n_chem: N_chem=4 unsupported at milestone 1 (only single-food N_chem=1 implemented); clamping to 1.


## 2. Run the N_chem=1 build on the fixed seed and compare to golden
The initial condition exercises eating, tax death, reproduction, and
LUT+egene mutation (~2300 of 2304 cells alive, heavy genome churn), so the
bit-exact comparison is a strong gate.

In [3]:
golden = cp.load_golden()
got    = cp.run_and_capture(EvoCA())
ok, lines = cp.compare(golden, got)
for ln in lines:
    print(ln)
print()
print('=' * 40)
print('  N_chem=1 PARITY:', 'PASS' if ok else 'FAIL')
print('=' * 40)
assert ok, 'N_chem=1 diverged from single-food golden'

  f         OK       (identical, 9216 values)
  F         OK       (identical, 9216 values)
  alive     OK       (identical, 9216 values)
  lut_hash  OK       (identical, 9216 values)

  N_chem=1 PARITY: PASS


## 3. Dynamics summary (the run is non-trivial)
For context: population and distinct-LUT counts at each checkpoint, from the
verified-identical run.

In [4]:
for i, cp_step in enumerate(cp.CHECKPOINTS):
    a = got['alive'][i]; f = got['f'][i]; h = got['lut_hash'][i]
    n_alive = int(a.sum())
    n_lut = len(np.unique(h[a == 1])) if n_alive else 0
    fmean = float(f[a == 1].mean()) if n_alive else 0.0
    print(f'step {cp_step:3d}:  alive={n_alive:4d}  distinct_LUT={n_lut:4d}'
          f'  f_mean={fmean:.3f}')

step  50:  alive=2288  distinct_LUT=2166  f_mean=0.670
step 100:  alive=2303  distinct_LUT=2180  f_mean=0.647
step 150:  alive=2300  distinct_LUT=2173  f_mean=0.635
step 200:  alive=2293  distinct_LUT=2170  f_mean=0.626


## Integration fix: `stamp_patch` keeps the chem store in sync

`stamp_patch` writes `f_priv` straight through the `evoca_get_f()` pointer. Without `evoca_chem_sync_from_f()` the next tax restores the host cell's old private food into the stamped organism (found during the Tier-2 merge, Oct 2026). Pre-chem behavior: a stamped cell starts at `f = 0` and is taxed to death on the next step.

In [5]:
N = 32
src = EvoCA(); src.init(N, food_inc=0.0, tax=0.01)
src.state(lut='gol', egenome='uniform', egenome_value=3)
src.set_f_all(0.5); src.set_alive_halfplane(0)
patch = src.extract_patch(0, 0, 8); src.free()
host = EvoCA(); host.init(N, food_inc=0.0, m_scale=0.0, tax=0.01)
host.state(lut='gol', egenome='uniform', egenome_value=3)
host.set_F_all(0.0); host.set_f_all(0.8)
host.stamp_patch(patch, 10, 10, mode='overwrite'); host.step()
n_alive = int(host.get_alive().reshape(N, N)[10:18, 10:18].sum())
print('stamped cells alive after 1 step:', n_alive, '(expected 0)')
host.free()
assert n_alive == 0

stamped cells alive after 1 step: 0 (expected 0)
